In [7]:
import os
import joblib
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [2]:
ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [3]:
data = load_iris()
X = data.data
y = data.target

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("iris_classification")

2026/09/02 22:15:39 INFO mlflow.tracking.fluent: Experiment with name 'iris_classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1788367539880, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788367539880, lifecycle_stage='active', name='iris_classification', tags={}, trace_location=None, workspace='default'>

In [8]:
def train_and_log(model, model_name, params):
    """Train a model and log everything to MLflow."""
    with mlflow.start_run(run_name=model_name):
        # Train
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
 
        # Metrics
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")
 
        # Log params and metrics
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
 
        # Log model the MLflow way (saved under the run's "model" artifact folder)
        mlflow.sklearn.log_model(model, artifact_path="model")
 
        # Also save a local .pkl copy and log it as an extra artifact
        local_path = os.path.join(ARTIFACT_DIR, f"{model_name}.pkl")
        joblib.dump(model, local_path)
        mlflow.log_artifact(local_path, artifact_path="pickled_model")
 
        print(f"{model_name} -> accuracy: {acc:.3f}, f1: {f1:.3f}")